#### RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

# Deprecated ..
# from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\localhost\AppData\Local\Temp\ipykernel_12864\4093396855.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [18]:
### Read all the pdf's inside the directory.

def process_all_pdfs( pdf_directory ):
    """ Process all PDF Files in a directory """

    all_documents = []
    pdf_dir = Path( pdf_directory )

    # Find all PDF files recursively
    pdf_files = list( pdf_dir.glob("**/*.pdf") )

    # print(pdf_files)

    print(f"Found { len(pdf_files) } PDF files to process.")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")

        try:
            loader = PyPDFLoader( str(pdf_file) )
            documents = loader.load()

            # So 1 document can contain multiple pages.
            # each pages contain its page_content and the metadata.
            print(f"> Document: {documents}")

            # documents (Array) => [ Document( metadata={}, page_content={}) , Document( metadata={}, page_content={}) ]
            # print(f"> 1st Page from Document#1: {documents[0]}")
            # print(f"> Metadata of 1st Page from Document#1: {documents[0].metadata}")

            # before adding extra metadata
            print(list(documents[0].metadata.keys()))

            # Add source information to METADATA
            for doc in documents:
                # so doc refers to each page of the documents
                # for each page of the document, add the metadata
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            # after adding extra metadata
            print(list(documents[0].metadata.keys()))

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages.")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\nTotal documents loaded: { len(all_documents) }")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")
all_pdf_documents

Found 2 PDF files to process.

Processing: 05. Free APIs for Project.pdf
> Document: [Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2023-08-08T20:35:13+05:30', 'title': 'Free APIs for Project', 'author': 'shiv', 'moddate': '2023-08-08T20:35:13+05:30', 'source': '..\\data\\pdf\\05. Free APIs for Project.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Free APIs for Project \n1. Potter DB \nDocs - https://potterdb.com/?ref=apislist.com \nAPI - https://api.potterdb.com/ \n2. Random Cats \nDocs - https://cataas.com/doc.html \nAPI - https://cataas.com \n3. Reddit \nDocs - https://www.reddit.com/dev/api/ \nAPI - https://www.reddit.com/r/Wallstreetbets/top.json \n4. Food Facts \nDocs - https://wiki.openfoodfacts.org/API \nAPI - https://world.openfoodfacts.org/api/v0/product/737628064502.json \n5. Random Jokes \nDocs - https://sv443.net/jokeapi/v2/ \nAPI - https://v2.jokeapi.dev/joke/Any?safe-mode \n6. Dictionary 

[Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2023-08-08T20:35:13+05:30', 'title': 'Free APIs for Project', 'author': 'shiv', 'moddate': '2023-08-08T20:35:13+05:30', 'source': '..\\data\\pdf\\05. Free APIs for Project.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': '05. Free APIs for Project.pdf', 'file_type': 'pdf'}, page_content='Free APIs for Project \n1. Potter DB \nDocs - https://potterdb.com/?ref=apislist.com \nAPI - https://api.potterdb.com/ \n2. Random Cats \nDocs - https://cataas.com/doc.html \nAPI - https://cataas.com \n3. Reddit \nDocs - https://www.reddit.com/dev/api/ \nAPI - https://www.reddit.com/r/Wallstreetbets/top.json \n4. Food Facts \nDocs - https://wiki.openfoodfacts.org/API \nAPI - https://world.openfoodfacts.org/api/v0/product/737628064502.json \n5. Random Jokes \nDocs - https://sv443.net/jokeapi/v2/ \nAPI - https://v2.jokeapi.dev/joke/Any?safe-mode \n6. Dictionary \nDocs - https://

In [19]:
### Text Splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance."""

    # chunk_overlap -> some text will be overlapped b/w two different documents

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n', '\n', ' ', '']
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    # Show example of a chunk
    if split_docs:
        print("\nExample Chunk:")
        print(f"Content: {split_docs[0].page_content[:50]}....")
        print(f"Content: {split_docs[0].page_content[:200]}....")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

chunks = split_documents(all_pdf_documents)

Split 37 documents into 56 chunks.

Example Chunk:
Content: Free APIs for Project 
1. Potter DB 
Docs - https:....
Content: Free APIs for Project 
1. Potter DB 
Docs - https://potterdb.com/?ref=apislist.com 
API - https://api.potterdb.com/ 
2. Random Cats 
Docs - https://cataas.com/doc.html 
API - https://cataas.com 
3. Re....
Metadata: {'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2023-08-08T20:35:13+05:30', 'title': 'Free APIs for Project', 'author': 'shiv', 'moddate': '2023-08-08T20:35:13+05:30', 'source': '..\\data\\pdf\\05. Free APIs for Project.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': '05. Free APIs for Project.pdf', 'file_type': 'pdf'}


#### Embedding and VectorStoreDB

In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

d:\RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer OpSource model."""

    def __init__(self, model_name: str ='all-MiniLM-L6-v2'):
        """
        Initialize the embedding manger

        Args:
            model_name: HuggingFace model name for sentence embeddings
        
        HF Link:
            https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    # underscore => Protected Function
    def _load_model(self):
        """Load the SentenceTransfomer model"""

        try:
            print(f"Loading Embedding Model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded Successfully. Embedding dimesion: {self.model.get_embedding_dimension()}")

            # get_embedding_dimension() -> default is 284 dimensions.
            # every text will be converted into 384 dimensions.
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts: List of text strings to embed.
        
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not loaded.")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f"Generate embeddings with shape: {embeddings.shape}")
        return embeddings

## Initialize EmbeddingManager

embeddingmanager = EmbeddingManager()
embeddingmanager

Loading Embedding Model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 620.78it/s]


Model loaded Successfully. Embedding dimesion: 384


In [21]:
class VectorStore:
    """Manages document embeddings in a ChromaDB Vector Store"""

    def __init__(self, collection_name: str="pdf_documents", persist_directory: str ="../data/vector_store"):
        """
        Initialize the Vector Store

        Args:
            collection_name: Name the ChromaDB collection
            persist_directory: Directory to persist the vector store -> storing the vectorstore in the local disk
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()


    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)

            # Get or Create Collection
            self.collection = self.client.get_or_create_collection(
                name= self.collection_name,
                metadata= {"description": "PDF Document Embeddings for RAG."}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e: 
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add Documents and their embeddings to the vector store

        Args:
            documents: List of Langchain Documents
            embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store.")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        print(zip(documents, embeddings)) # tuple

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate the Unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare Metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents = documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 270


### Convert the text to embeddings


In [22]:
texts = [doc.page_content for doc in chunks]
print(len(texts))
texts

56


['Free APIs for Project \n1. Potter DB \nDocs - https://potterdb.com/?ref=apislist.com \nAPI - https://api.potterdb.com/ \n2. Random Cats \nDocs - https://cataas.com/doc.html \nAPI - https://cataas.com \n3. Reddit \nDocs - https://www.reddit.com/dev/api/ \nAPI - https://www.reddit.com/r/Wallstreetbets/top.json \n4. Food Facts \nDocs - https://wiki.openfoodfacts.org/API \nAPI - https://world.openfoodfacts.org/api/v0/product/737628064502.json \n5. Random Jokes \nDocs - https://sv443.net/jokeapi/v2/ \nAPI - https://v2.jokeapi.dev/joke/Any?safe-mode \n6. Dictionary \nDocs - https://dictionaryapi.dev/ \nAPI - https://api.dictionaryapi.dev/api/v2/entries/en/digital \n7. Quotes \nDocs - https://pprathameshmore.github.io/QuoteGarden/ \nAPI - https://quote-garden.onrender.com/api/v3/quotes \n8. Library \nDocs - https://openlibrary.org/developers/api \nAPI - http://openlibrary.org/api/volumes/brief/isbn/9780525440987.json',
 '9. Generates Fake Data \nDocs - https://fakerapi.it/en \nAPI - https:/

In [23]:
# Generate the embeddings
embedding = embeddingmanager.generate_embeddings(texts)
# embedded output of a first text
print(embedding[0])
#  (len(texts), embedding_dim)

Generating embeddings for 56 texts...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches: 100%|██████████| 2/2 [00:05<00:00,  2.89s/it]

Generate embeddings with shape: (56, 384)
[-7.15018585e-02 -3.91998366e-02 -5.97780459e-02  2.49811243e-02
  7.55666243e-03 -4.27288301e-02 -5.93372211e-02 -2.96629081e-03
 -4.78570797e-02 -7.39196141e-04  5.13649024e-02 -4.26288433e-02
  3.93976048e-02  2.12617801e-04  7.59751201e-02  2.27061473e-02
 -4.61819442e-03  6.78079575e-02 -8.76245555e-03 -8.72346237e-02
  7.76911080e-02  3.82299162e-02  1.00967370e-01 -3.75126414e-02
 -2.42827721e-02 -8.00230131e-02 -4.38379943e-02 -5.55159757e-03
 -1.47629827e-02 -2.19144430e-02 -3.44664752e-02  9.98599362e-03
 -1.88975711e-03 -8.18764139e-03  1.51429661e-02  8.26467481e-03
  5.16314209e-02 -7.66833648e-02  3.23031507e-02  3.00417785e-02
 -4.42969501e-02  6.24653359e-04 -1.24586008e-01 -2.30733305e-02
  1.39952740e-02  7.14474358e-03 -1.03653744e-01  1.09762317e-02
 -2.42170878e-02  5.69061264e-02 -9.39611793e-02 -5.73005415e-02
 -4.28326093e-02 -2.54352242e-02  7.76006281e-02 -2.86565758e-02
 -4.09627110e-02  3.96455312e-03 -8.59725401e-02

In [24]:

# Store in Vector DB
vectorstore.add_documents(chunks, embedding)

Adding 56 documents to vector store.
Successfully added 56 documents to vector store
Total documents in collection: 326


### Retriever Pipeline From Vector Store

In [25]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector Store containing the document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query.

        Args:
            query: The search query
            top_k: Number of top results to return for the query
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metdata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate Query Embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )

            print(results)
            print(results['documents'])

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1- distance

                    #zip line

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")

            else:
                print("No documents found.")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            raise

In [26]:
rag_retriever = RAGRetriever(vectorstore, embeddingmanager)


In [27]:
query = "Which API is used to generate 'Random Jokes' ?"
rag_retriever.retrieve(query = query)

Retrieving documents for query: 'Which API is used to generate 'Random Jokes' ?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.14it/s]

Generate embeddings with shape: (1, 384)
{'ids': [['doc_4399465f_0', 'doc_df407235_1', 'doc_bcc15257_30', 'doc_55f63e30_48', 'doc_a82d2f9a_32']], 'embeddings': None, 'documents': [['Free APIs for Project \n1. Potter DB \nDocs - https://potterdb.com/?ref=apislist.com \nAPI - https://api.potterdb.com/ \n2. Random Cats \nDocs - https://cataas.com/doc.html \nAPI - https://cataas.com \n3. Reddit \nDocs - https://www.reddit.com/dev/api/ \nAPI - https://www.reddit.com/r/Wallstreetbets/top.json \n4. Food Facts \nDocs - https://wiki.openfoodfacts.org/API \nAPI - https://world.openfoodfacts.org/api/v0/product/737628064502.json \n5. Random Jokes \nDocs - https://sv443.net/jokeapi/v2/ \nAPI - https://v2.jokeapi.dev/joke/Any?safe-mode \n6. Dictionary \nDocs - https://dictionaryapi.dev/ \nAPI - https://api.dictionaryapi.dev/api/v2/entries/en/digital \n7. Quotes \nDocs - https://pprathameshmore.github.io/QuoteGarden/ \nAPI - https://quote-garden.onrender.com/api/v3/quotes \n8. Library \nDocs - https:

[{'id': 'doc_4399465f_0',
  'content': 'Free APIs for Project \n1. Potter DB \nDocs - https://potterdb.com/?ref=apislist.com \nAPI - https://api.potterdb.com/ \n2. Random Cats \nDocs - https://cataas.com/doc.html \nAPI - https://cataas.com \n3. Reddit \nDocs - https://www.reddit.com/dev/api/ \nAPI - https://www.reddit.com/r/Wallstreetbets/top.json \n4. Food Facts \nDocs - https://wiki.openfoodfacts.org/API \nAPI - https://world.openfoodfacts.org/api/v0/product/737628064502.json \n5. Random Jokes \nDocs - https://sv443.net/jokeapi/v2/ \nAPI - https://v2.jokeapi.dev/joke/Any?safe-mode \n6. Dictionary \nDocs - https://dictionaryapi.dev/ \nAPI - https://api.dictionaryapi.dev/api/v2/entries/en/digital \n7. Quotes \nDocs - https://pprathameshmore.github.io/QuoteGarden/ \nAPI - https://quote-garden.onrender.com/api/v3/quotes \n8. Library \nDocs - https://openlibrary.org/developers/api \nAPI - http://openlibrary.org/api/volumes/brief/isbn/9780525440987.json',
  'metadata': {'creationdate': '20

In [29]:
query = "What is the purpose of Reinforcement Learning ?"
rag_retriever.retrieve(query = query)

Retrieving documents for query: 'What is the purpose of Reinforcement Learning ?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.84it/s]

Generate embeddings with shape: (1, 384)
{'ids': [['doc_19509d95_20', 'doc_85ee167e_25', 'doc_d49f12ca_24', 'doc_b259aa11_23', 'doc_9cb15705_26']], 'embeddings': None, 'documents': [["Finally, we get to something looks like real artificial intelligence. In \nlots of articles reinforcement learning is placed somewhere in \nbetween of supervised and unsupervised learning. They have nothing \nin common! Is this because of the name?\nReinforcement learning is used in cases when your problem is not \nrelated to data at all, but you have an environment to live in. Like a \nvideo game world or a city for self-driving car.\n…\nNeural network plays Mario\nKnowledge of all the road rules in the world will not teach the \nautopilot how to drive on the roads. Regardless of how much data we \ncollect, we still can't foresee all the possible situations. This is why its \ngoal is to minimize error, not to predict all the moves.\nSurviving in an environment is a core idea of reinforcement learning. \n

[{'id': 'doc_19509d95_20',
  'content': "Finally, we get to something looks like real artificial intelligence. In \nlots of articles reinforcement learning is placed somewhere in \nbetween of supervised and unsupervised learning. They have nothing \nin common! Is this because of the name?\nReinforcement learning is used in cases when your problem is not \nrelated to data at all, but you have an environment to live in. Like a \nvideo game world or a city for self-driving car.\n…\nNeural network plays Mario\nKnowledge of all the road rules in the world will not teach the \nautopilot how to drive on the roads. Regardless of how much data we \ncollect, we still can't foresee all the possible situations. This is why its \ngoal is to minimize error, not to predict all the moves.\nSurviving in an environment is a core idea of reinforcement learning. \nThrow poor little robot into real life, punish it for errors and reward \nit for right deeds. Same way we teach our kids, right?",
  'metadata'

#### Integration VectorDB Context Pipeline with LLM Output.

In [41]:
# Simple RAG Pipeline with Groq LLM

import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv('GROQ_API_KEY')

llm = ChatGroq(
    groq_api_key = groq_api_key,
    model_name = 'openai/gpt-oss-20b',
    temperature = 0.1,
    max_tokens = 1024
)

# Simple RAG Function : Retrieve Context + Generate Response
def rag_simple(query, retriever, llm, top_k = 3):

    # retrieve the context
    results = retriever.retrieve(query, top_k = top_k)

    context = "\n\n".join( [doc['content'] for doc in results] ) if results else ""

    if not context:
        return "No relevant context found."

    # generate the Output from the LLM
    prompt = f"""Use the following context to answer the question concisely.

    Context:
    {context}

    Query: {query}

    Answer:"""

    response = llm.invoke(
        [ prompt.format(context= context, query = query) ]
    )
    return response.content


In [44]:
answer = rag_simple("What is the purpose of Reinforcement Learning ?",
                    rag_retriever,
                    llm)

print("Answer:\n")
print(answer)

Retrieving documents for query: 'What is the purpose of Reinforcement Learning ?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  5.40it/s]


Generate embeddings with shape: (1, 384)
{'ids': [['doc_19509d95_20', 'doc_85ee167e_25', 'doc_d49f12ca_24']], 'embeddings': None, 'documents': [["Finally, we get to something looks like real artificial intelligence. In \nlots of articles reinforcement learning is placed somewhere in \nbetween of supervised and unsupervised learning. They have nothing \nin common! Is this because of the name?\nReinforcement learning is used in cases when your problem is not \nrelated to data at all, but you have an environment to live in. Like a \nvideo game world or a city for self-driving car.\n…\nNeural network plays Mario\nKnowledge of all the road rules in the world will not teach the \nautopilot how to drive on the roads. Regardless of how much data we \ncollect, we still can't foresee all the possible situations. This is why its \ngoal is to minimize error, not to predict all the moves.\nSurviving in an environment is a core idea of reinforcement learning. \nThrow poor little robot into real life

In [ ]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """

    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])

    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]

    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is the purpose of Reinforcement Learning ?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is the purpose of Reinforcement Learning ?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.36s/it]


Generate embeddings with shape: (1, 384)
{'ids': [['doc_19509d95_20', 'doc_85ee167e_25', 'doc_d49f12ca_24']], 'embeddings': None, 'documents': [["Finally, we get to something looks like real artificial intelligence. In \nlots of articles reinforcement learning is placed somewhere in \nbetween of supervised and unsupervised learning. They have nothing \nin common! Is this because of the name?\nReinforcement learning is used in cases when your problem is not \nrelated to data at all, but you have an environment to live in. Like a \nvideo game world or a city for self-driving car.\n…\nNeural network plays Mario\nKnowledge of all the road rules in the world will not teach the \nautopilot how to drive on the roads. Regardless of how much data we \ncollect, we still can't foresee all the possible situations. This is why its \ngoal is to minimize error, not to predict all the moves.\nSurviving in an environment is a core idea of reinforcement learning. \nThrow poor little robot into real life

In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:

        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]

            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("What is the purpose of Reinforcement Learning ?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'Why Machine Learning is required?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.49it/s]

Generate embeddings with shape: (1, 384)
{'ids': [['doc_2c09d445_4', 'doc_72b7a40d_5', 'doc_f86216a6_3']], 'embeddings': None, 'documents': [['The map of the machine learning \nworld\nPart Ͽ. Classical Machine Learning\nThe first methods came from pure statistics in the \'ЃϾs. They solved \nformal math tasks — searching for patterns in numbers, evaluating \nthe proximity of data points, and calculating vectors\' directions.\nNowadays, half of the Internet is working on these algorithms. When \nyou see a list of articles to "read next" or your bank blocks your card', "at random gas station in the middle of nowhere, most likely it's the \nwork of one of those little guys.\nBig tech companies are huge fans of neural networks. Obviously. For \nthem, Ѐ% accuracy is an additional Ѐ billion in revenue. But when you \nare small, it doesn't make sense. I heard stories of the teams spending \na year on a new recommendation algorithm for their e-commerce \nwebsite, before discovering that 99% of 

ng for patterns in numbers, evaluating 
the proximity of data points, and calculating vectors' directions.
Nowadays, half of the Internet is working on these algorithms. When 
you see a list of articles to "read next" or your bank blocks your card

at random gas station in the middle of nowhere, most likely it's the 
work of one of those little guys.
Big tech companies are huge fans of neural networks. Obviously. For 
them, Ѐ% accuracy is an additional Ѐ billion in revenue. But when you 
are small, it doesn't make sense. I heard stories of the teams spending 
a year on a new recommendation algorithm for their e-commerce 
website, before discovering that 99% of traffic came from search 
engines. Their algorithms were useless. Most users didn't even open 
the main page.
Despite the popularity, classical approaches are so natural that you 
could easily explain them to a toddler. They are like basic arithmetic 
— we use it every day, without even thinking.
[collapse all][show all]Ѐ comment